Implementation of the Mamba Architecture

In [1]:
import tiktoken
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import torch.nn.functional as F
import urllib.request as req

In [2]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

In [3]:
try:
    with req.urlopen(url) as response:
        raw_text = response.read().decode('utf-8')
    print("Total Length", len(raw_text))
except:
    print("Text download failed")

Total Length 1115394


In [4]:
corpus = raw_text[:150]
print("Corpus Excerpt: \n", "="*30, "\n", corpus)

Corpus Excerpt: 
 First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

A


In [5]:
enc = tiktoken.encoding_for_model("gpt-4o")
token_ids = enc.encode(raw_text)
vocab_size = enc.max_token_value + 1

In [6]:
data_tensor = torch.tensor(token_ids, dtype=torch.long)

Parameters

In [7]:
block_size = 128
batch_size = 16
d_model = 128
d_state = 16
n_layers = 4
eps = 1e-6
learning_rate = 1e-3
max_iters = 200
eval_interval = 100
d_conv = 4
expand_factor = 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Train/Val Split

In [8]:
n = int(0.9 * len(data_tensor))
train_data = data_tensor[:n]
val_data = data_tensor[n:]

Batching

In [9]:
def get_batch(split, device):
    data = train_data if split == "train" else val_data
    ix = torch.randint(0, len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

Verifying Batching

In [10]:
x, y = get_batch("train", device)
assert x.shape == (batch_size, block_size)
assert y.shape == (batch_size, block_size)

Root Mean Square Normalisation

In [11]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight

In [12]:
# Test RMS
rms = RMSNorm(d_model, eps)
input_tensor = torch.randn([batch_size, block_size, d_model])
rms_test_res = rms(input_tensor)
assert input_tensor.shape == rms_test_res.shape

Mamba Block

In [13]:
class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state, d_conv, expand_factor):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_conv = d_conv
        self.expand_factor = expand_factor
        self.inner_dim = d_model * expand_factor

        self.rmsnorm = RMSNorm(d_model)
        # Matrix B
        self.in_proj = nn.Linear(d_model, 2*self.inner_dim, bias=False)
        # Matrix C
        self.out_proj = nn.Linear(self.inner_dim, d_model, bias=False)

        self.conv1d = nn.Conv1d(
            in_channels=self.inner_dim,
            out_channels=self.inner_dim,
            kernel_size=self.d_conv,
            groups=self.inner_dim,
            bias=True
        )
    
    def forward(self, x):
        normed_x = self.rmsnorm(x)
        projected_x = self.in_proj(normed_x)
        x_stream, z_gate = projected_x.chunk(2, dim=-1)
        # Conv Ops
        x_stream = x_stream.transpose(1, 2)
        x_stream = F.pad(x_stream, (self.d_conv - 1, 0))
        x_conv = self.conv1d(x_stream)
        x_conv = F.silu(x_conv)
        x_stream = x_conv.transpose(1, 2)

        processed_x = self.out_proj(x_stream)
        return x + processed_x

Tiny Mamba

In [14]:
class TinyMamba(nn.Module):
    def __init__(self, vocab_size, d_model, d_state, d_conv, n_layers, eps, expand_factor):
        super().__init__()
        self.vocab_size = vocab_size
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([MambaBlock(d_model, d_state, d_conv, expand_factor) for _ in range(n_layers)])
        self.rmsnorm = RMSNorm(d_model, eps)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx, targets):
        x = self.token_embedding(idx)
        for block in self.blocks:
            x = block(x)
        x = self.rmsnorm(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            logits_flat = logits.view(-1, self.vocab_size)
            targets_flat = targets.view(-1)
            loss = F.cross_entropy(logits_flat, targets_flat)
        return logits, loss

In [15]:
# Test run
# test_model = TinyMamba(vocab_size, d_model, d_state, d_conv, n_layers, eps, expand_factor).to(device)
# test_optimizer = torch.optim.AdamW(test_model.parameters(), lr=learning_rate)
# test_logits, test_loss = test_model(x, y)
# test_optimizer.zero_grad()
# test_loss.backward()
# test_optimizer.step()

# assert test_logits.shape == (batch_size, block_size, vocab_size)
# assert test_loss.ndim == 0
# assert torch.isfinite(test_loss)

Training and Evaluation

In [16]:
# Estimating Loss
@torch.no_grad
def estimate_loss(model, device, eval_interval):
    out = {}
    model.eval()

    for split in ["train", "val"]:
        losses = torch.zeros(eval_interval)
        for k in range(eval_interval):
            x_batch, y_batch = get_batch(split, device)
            logits, loss = model(x_batch, y_batch)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

In [ ]:
model = TinyMamba(vocab_size, d_model, d_state, d_conv, n_layers, eps, expand_factor).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
train_loss_history, val_loss_history = {}, {}

for step in range(max_iters):
    x_batch, y_batch = get_batch("train", device)
    logits, loss = model(x_batch, y_batch)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % eval_interval == 0:
        losses = estimate_loss(model, device, eval_interval)
        train_loss_history[step] = losses["train"]
        val_loss_history[step] = losses["val"]
        print(f"Step {step:4d} | Train Loss: {losses['train']:.5f} | Val Loss: {losses['val']:.5f}")